In [ ]:

import os, glob, cv2, random
import numpy as np
from PIL import Image
from pathlib import Path
from facenet_pytorch import MTCNN
import torch
from tqdm import tqdm

FFPP_ROOT = r"E:\FaceForensics++" 
OUT_ROOT  = r"E:\SFIAD_Project\ffpp_fused_npy"

REAL_DIR = os.path.join(
    FFPP_ROOT,
    "original_sequences", "youtube", "c23", "videos"
)

FAKE_DIRS = [
    os.path.join(FFPP_ROOT, "manipulated_sequences", "Deepfakes", "c23", "videos"),
    os.path.join(FFPP_ROOT, "manipulated_sequences", "Face2Face", "c23", "videos"),
    os.path.join(FFPP_ROOT, "manipulated_sequences", "FaceSwap", "c23", "videos"),
    os.path.join(FFPP_ROOT, "manipulated_sequences", "NeuralTextures", "c23", "videos"),
]

TRAIN_RATIO = 0.90
SEED = 42

FRAME_STEP = 32
IMAGE_SIZE = 256
MARGIN = 20
DTYPE = np.float16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def ensure_dir(p):
    Path(p).mkdir(parents=True, exist_ok=True)


def collect_videos(folder):
    if not os.path.isdir(folder):
        return []
    return sorted(glob.glob(os.path.join(folder, "*.mp4")))


def split_list(lst, ratio=0.9):
    random.shuffle(lst)
    n = int(len(lst) * ratio)
    return lst[:n], lst[n:]


def compute_fft(gray):
    f = np.fft.fft2(gray)
    f = np.fft.fftshift(f)
    mag = np.abs(f)
    mag = np.log1p(mag)
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag.astype(np.float32)


def process_video(video_path, out_dir, mtcnn):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return

    base = os.path.splitext(os.path.basename(video_path))[0]

    idx = 0
    saved = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if idx % FRAME_STEP == 0:

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(rgb)

            with torch.no_grad():
                face = mtcnn(img)

            if face is None:
                idx += 1
                continue

            face = (face * 255).clamp(0,255).byte()
            face = face.permute(1,2,0).cpu().numpy()

            face = Image.fromarray(face).resize((IMAGE_SIZE, IMAGE_SIZE))

            rgb_arr = np.asarray(face, dtype=np.float32) / 255.0

            gray = rgb_arr.mean(axis=2)

            fft = compute_fft(gray)

            fused = np.dstack([rgb_arr, fft]).astype(DTYPE)

            save_name = f"{base}_f{saved:05d}.npy"

            np.save(os.path.join(out_dir, save_name), fused)

            saved += 1

        idx += 1

    cap.release()


def run_split(video_list, out_dir, mtcnn, desc):
    ensure_dir(out_dir)

    for vp in tqdm(video_list, desc=desc, ncols=80):
        process_video(vp, out_dir, mtcnn)


def main():

    random.seed(SEED)

    print("FF++ RGB+FFT Extraction")
    print("Device:", DEVICE)

    mtcnn = MTCNN(
        image_size=IMAGE_SIZE,
        margin=MARGIN,
        keep_all=False,
        device=DEVICE,
        post_process=True
    )

    real_videos = collect_videos(REAL_DIR)

    train_real, test_real = split_list(real_videos, TRAIN_RATIO)

    print("REAL Total :", len(real_videos))
    print("Train REAL :", len(train_real))
    print("Test REAL  :", len(test_real))

    run_split(
        train_real,
        os.path.join(OUT_ROOT, "train", "real"),
        mtcnn,
        "TRAIN REAL"
    )

    run_split(
        test_real,
        os.path.join(OUT_ROOT, "test", "real"),
        mtcnn,
        "TEST REAL"
    )

    fake_videos = []

    for d in FAKE_DIRS:
        vids = collect_videos(d)
        fake_videos.extend(vids)

    train_fake, test_fake = split_list(fake_videos, TRAIN_RATIO)

    print("FAKE Total :", len(fake_videos))
    print("Train FAKE :", len(train_fake))
    print("Test FAKE  :", len(test_fake))

    run_split(
        train_fake,
        os.path.join(OUT_ROOT, "train", "fake"),
        mtcnn,
        "TRAIN FAKE"
    )

    run_split(
        test_fake,
        os.path.join(OUT_ROOT, "test", "fake"),
        mtcnn,
        "TEST FAKE"
    )

    print("\nDONE")
    print("Saved to:", OUT_ROOT)


if __name__ == "__main__":
    main()